In [1]:
import pandas as pd
import numpy as np
import re
import string
from nltk.stem import WordNetLemmatizer

from nltk.corpus import stopwords

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report

import mlflow
import mlflow.sklearn

In [2]:
# import nltk

# nltk.download('wordnet')
# nltk.download('omw-1.4')
# nltk.download('stopwords')

In [3]:
# import sys
# from pathlib import Path

# ROOT_DIR = Path().resolve().parent
# sys.path.append(str(ROOT_DIR))

# from src.config_mlflow import setup_mlflow

# setup_mlflow()

In [4]:
# mlflow.set_experiment("ML_Model_Ticket_Classification")

In [ ]:
df=pd.read_csv('../data/customer_support_tickets.csv')

In [6]:
df.head()

,Ticket ID,Customer Name,Customer Email,Customer Age,Customer Gender,Product Purchased,Date of Purchase,Ticket Type,Ticket Subject,Ticket Description,Ticket Status,Resolution,Ticket Priority,Ticket Channel,First Response Time,Time to Resolution,Customer Satisfaction Rating
0,1,Marisa Obrien,carrollallison@example.com,32,Other,GoPro Hero,2021-03-22,Technical issue,Product setup,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Social media,2023-06-01 12:15:36,NaN,NaN
1,2,Jessica Rios,clarkeashley@example.com,42,Female,LG Smart TV,2021-05-22,Technical issue,Peripheral compatibility,I'm having an issue with the {product_purchase...,Pending Customer Response,NaN,Critical,Chat,2023-06-01 16:45:38,NaN,NaN
2,3,Christopher Robbins,gonzalestracy@example.com,48,Other,Dell XPS,2020-07-14,Technical issue,Network problem,I'm facing a problem with my {product_purchase...,Closed,Case maybe show recently my computer follow.,Low,Social media,2023-06-01 11:14:38,2023-06-01 18:05:38,3.0
3,4,Christina Dillon,bradleyolson@example.org,27,Female,Microsoft Office,2020-11-13,Billing inquiry,Account access,I'm having an issue with the {product_purchase...,Closed,Try capital clearly never color toward story.,Low,Social media,2023-06-01 07:29:40,2023-06-01 01:57:40,3.0
4,5,Alexander Carroll,bradleymark@example.com,67,Female,Autodesk AutoCAD,2020-02-04,Billing inquiry,Data loss,I'm having an issue with the {product_purchase...,Closed,West decision evidence bit.,Low,Email,2023-06-01 00:12:42,2023-06-01 19:53:42,1.0


In [7]:
df=df[['Ticket Subject','Ticket Description','Ticket Type']]

In [8]:
df.head()

,Ticket Subject,Ticket Description,Ticket Type
0,Product setup,I'm having an issue with the {product_purchase...,Technical issue
1,Peripheral compatibility,I'm having an issue with the {product_purchase...,Technical issue
2,Network problem,I'm facing a problem with my {product_purchase...,Technical issue
3,Account access,I'm having an issue with the {product_purchase...,Billing inquiry
4,Data loss,I'm having an issue with the {product_purchase...,Billing inquiry


In [9]:
max_len = df['Ticket Description'].str.len().max()
max_len

397

In [10]:
pd.set_option('display.max_colwidth', None)

In [11]:
df.head()

,Ticket Subject,Ticket Description,Ticket Type
0,Product setup,"I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.",Technical issue
1,Peripheral compatibility,"I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. Please assist.\n\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",Technical issue
2,Network problem,"I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\n\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it's not charging properly.",Technical issue
3,Account access,"I'm having an issue with the {product_purchased}. Please assist.\n\nIf you have a problem you're interested in and I'd love to see this happen, please check out the Feedback. I've already contacted customer support multiple times, but the issue remains unresolved.",Billing inquiry
4,Data loss,I'm having an issue with the {product_purchased}. Please assist.\n\n\nNote: The seller is not responsible for any damages arising out of the delivery of the battleground game. Please have the game in good condition and shipped to you I've noticed a sudden decrease in battery life on my {product_purchased}. It used to last much longer.,Billing inquiry


In [12]:
df.tail()

,Ticket Subject,Ticket Description,Ticket Type
8464,Installation support,"My {product_purchased} is making strange noises and not functioning properly. I suspect there might be a hardware issue. Can you please help me with this?\n\nAs always, you can email me at support@laserprint. I need assistance as soon as possible because it's affecting my work and productivity.",Product inquiry
8465,Refund request,"I'm having an issue with the {product_purchased}. Please assist.\n\n\nSell for 30-35% (I also bought a 2.5mm and I'm getting stuck).\n\n\nBuy for 80-100% ( The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.",Technical issue
8466,Account access,"I'm having an issue with the {product_purchased}. Please assist. You are using a different browser than the one I'm using. I've performed a factory reset on my {product_purchased}, hoping it would resolve the problem, but it didn't help.",Technical issue
8467,Payment issue,"I'm having an issue with the {product_purchased}. Please assist. I don't think a product is in the same category as an {product_purchased}, as it wouldn't have been the issue in the case of the I've noticed that the issue occurs consistently when I use a specific feature or application on my {product_purchased}.",Product inquiry
8468,Hardware issue,"There seems to be a hardware problem with my {product_purchased}. The screen is flickering, and I'm unable to use it. What should I do? I'll try my best to fix it now.\n\nI could get a I've reviewed the troubleshooting steps on the official support website, but they didn't resolve the problem.",Billing inquiry


In [13]:
print(df['Ticket Description'].head())

0                                                  I'm having an issue with the {product_purchased}. Please assist.\n\nYour billing zip code is: 71701.\n\nWe appreciate that you have requested a website address.\n\nPlease double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.
1                                                    I'm having an issue with the {product_purchased}. Please assist.\n\nIf you need to change an existing product.\n\nI'm having an issue with the {product_purchased}. Please assist.\n\nIf The issue I'm facing is intermittent. Sometimes it works fine, but other times it acts up unexpectedly.
2                                                               I'm facing a problem with my {product_purchased}. The {product_purchased} is not turning on. It was working fine until yesterday, but now it doesn't respond.\n\n1.8.3 I really I'm using the original charger that came with my {product_purchased}, but it

In [14]:

def clean_text(text):
    lemmatizer= WordNetLemmatizer()
    stop_words=set(stopwords.words('english'))
    # Lowercase
    text = text.lower()

    # Remove numbers
    text = re.sub(r'\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Tokenization
    words = text.split()

    # Remove stopwords + lemmatization
    cleaned_words = []

    for word in words:

        # Remove stopwords
        if word not in stop_words:

            # Lemmatize
            lemma_word = lemmatizer.lemmatize(word)

            cleaned_words.append(lemma_word)

    # Join back
    text = " ".join(cleaned_words)

    return text

In [15]:
df['Full Text'] = df['Ticket Subject'].apply(clean_text)+" "+df['Ticket Description'].apply(clean_text)

# Let's see what a row looks like before and after
print("BEFORE:\n", df['Ticket Description'].iloc[0])
print("\nAFTER:\n", df['Full Text'].iloc[0])

BEFORE:
 I'm having an issue with the {product_purchased}. Please assist.

Your billing zip code is: 71701.

We appreciate that you have requested a website address.

Please double check your email address. I've tried troubleshooting steps mentioned in the user manual, but the issue persists.

AFTER:
 product setup im issue productpurchased please assist billing zip code appreciate requested website address please double check email address ive tried troubleshooting step mentioned user manual issue persists


In [16]:
from sklearn.preprocessing import LabelEncoder
target_encoder = LabelEncoder()

# Fit and transform your 'ticket type' column into numerical integers
df['Target_Encoded'] = target_encoder.fit_transform(df['Ticket Type'])

# Let's see the mapping to understand what it did
mapping = dict(zip(target_encoder.classes_, target_encoder.transform(target_encoder.classes_)))
print("Category Mapping Reference:")
print(mapping)

Category Mapping Reference:
{'Billing inquiry': 0, 'Cancellation request': 1, 'Product inquiry': 2, 'Refund request': 3, 'Technical issue': 4}


In [17]:
X = df['Full Text']
y = df['Target_Encoded']

In [18]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, 
random_state=42,stratify=y)

In [ ]:
tfidf = TfidfVectorizer(max_features=10000,ngram_range=(1,2),
                        min_df=2,
                        max_df=0.95)
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [20]:
X_train_tfidf=X_train_tfidf.astype('float32', copy=False)
X_test_tfidf=X_test_tfidf.astype('float32', copy=False)

In [21]:
from sklearn.metrics import accuracy_score, f1_score, classification_report
log=LogisticRegression(
    max_iter=2000,
    class_weight='balanced'
)
log.fit(X_train_tfidf,y_train)
predictions=log.predict(X_test_tfidf)
accuracy = accuracy_score(y_test, predictions)
f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)

print(f"Accuracy: {accuracy:.4f}")
print(f"F1-Score (Weighted): {f1_weighted:.4f}")
print(f"F1-Score (Macro): {f1_macro:.4f}\n")

Accuracy: 0.2048
F1-Score (Weighted): 0.2043
F1-Score (Macro): 0.2046



In [22]:
abc

NameError: name 'abc' is not defined

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report

model_name = "Decision_Tree_2"

with mlflow.start_run(run_name=model_name):
    # Initialize and train
    dt_model = DecisionTreeClassifier(max_depth=10, random_state=42)
    dt_model.fit(X_train_tfidf, y_train)
    
    # Predict
    predictions = dt_model.predict(X_test_tfidf)
    text_predictions = target_encoder.inverse_transform(predictions)
    text_y_test = target_encoder.inverse_transform(y_test)
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    
    # Log parameters to MLflow
    mlflow.log_param("max_depth", dt_model.max_depth)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    # Log all metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    
    # Save model artifact
    mlflow.sklearn.log_model(dt_model, artifact_path=model_name)
    
    print(f"--- {model_name} Results ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")
    print(classification_report(text_y_test, text_predictions, zero_division=0))

2026/05/30 18:17:47 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 18:17:47 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- Decision_Tree_2 Results ---
Accuracy: 0.2066
F1-Score (Weighted): 0.0836
F1-Score (Macro): 0.0815

                      precision    recall  f1-score   support

     Billing inquiry       0.20      0.00      0.01       327
Cancellation request       0.15      0.02      0.04       339
     Product inquiry       0.20      0.00      0.01       328
      Refund request       0.14      0.01      0.01       351
     Technical issue       0.21      0.97      0.34       349

            accuracy                           0.21      1694
           macro avg       0.18      0.20      0.08      1694
        weighted avg       0.18      0.21      0.08      1694

🏃 View run Decision_Tree_2 at: http://127.0.0.1:5000/#/experiments/2/runs/4fd49dba0d464cb3852dc0b80c845180
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
import mlflow.sklearn

model_name = "Logistic_Regression_2"

with mlflow.start_run(run_name=model_name):
    # Initialize and train
    lr_model = LogisticRegression(max_iter=1000, C=1.0)
    lr_model.fit(X_train_tfidf, y_train)
    
    # Predict and decode metrics back to text strings
    predictions = lr_model.predict(X_test_tfidf)
    text_predictions = target_encoder.inverse_transform(predictions)
    text_y_test = target_encoder.inverse_transform(y_test)
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    
    # Log parameters to MLflow
    mlflow.log_param("C", lr_model.C)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    # Log all metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    
    # Save model artifact
    mlflow.sklearn.log_model(lr_model, artifact_path=model_name)
    
    print(f"--- {model_name} Results ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")
    print(classification_report(text_y_test, text_predictions, zero_division=0))

2026/05/30 18:17:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 18:17:52 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- Logistic_Regression_2 Results ---
Accuracy: 0.2072
F1-Score (Weighted): 0.2072
F1-Score (Macro): 0.2070

                      precision    recall  f1-score   support

     Billing inquiry       0.20      0.19      0.20       327
Cancellation request       0.20      0.19      0.19       339
     Product inquiry       0.21      0.20      0.21       328
      Refund request       0.18      0.19      0.19       351
     Technical issue       0.25      0.26      0.25       349

            accuracy                           0.21      1694
           macro avg       0.21      0.21      0.21      1694
        weighted avg       0.21      0.21      0.21      1694

🏃 View run Logistic_Regression_2 at: http://127.0.0.1:5000/#/experiments/2/runs/8d2b158ddf5d4888859d1d64336dcae9
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [ ]:
from sklearn.naive_bayes import MultinomialNB

model_name = "Naive_Bayes_2"

with mlflow.start_run(run_name=model_name):
    # Initialize and train
    nb_model = MultinomialNB(alpha=1.0)
    nb_model.fit(X_train_tfidf, y_train)
    
    # Predict
    predictions = nb_model.predict(X_test_tfidf)
    text_predictions = target_encoder.inverse_transform(predictions)
    text_y_test = target_encoder.inverse_transform(y_test)
    
    # Calculate evaluation metrics
    accuracy = accuracy_score(y_test, predictions)
    f1_weighted = f1_score(y_test, predictions, average='weighted', zero_division=0)
    f1_macro = f1_score(y_test, predictions, average='macro', zero_division=0)
    
    # Log parameters to MLflow
    mlflow.log_param("alpha", nb_model.alpha)
    mlflow.log_param("vectorizer", "TF-IDF")
    
    # Log all metrics to MLflow
    mlflow.log_metric("accuracy", accuracy)
    mlflow.log_metric("f1_score_weighted", f1_weighted)
    mlflow.log_metric("f1_score_macro", f1_macro)
    
    # Save model artifact
    mlflow.sklearn.log_model(nb_model, artifact_path=model_name)
    
    print(f"--- {model_name} Results ---")
    print(f"Accuracy: {accuracy:.4f}")
    print(f"F1-Score (Weighted): {f1_weighted:.4f}")
    print(f"F1-Score (Macro): {f1_macro:.4f}\n")
    print(classification_report(text_y_test, text_predictions, zero_division=0))

2026/05/30 18:17:54 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/30 18:17:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


--- Naive_Bayes_2 Results ---
Accuracy: 0.1989
F1-Score (Weighted): 0.1975
F1-Score (Macro): 0.1969

                      precision    recall  f1-score   support

     Billing inquiry       0.20      0.15      0.17       327
Cancellation request       0.21      0.20      0.21       339
     Product inquiry       0.18      0.16      0.17       328
      Refund request       0.19      0.23      0.21       351
     Technical issue       0.21      0.24      0.23       349

            accuracy                           0.20      1694
           macro avg       0.20      0.20      0.20      1694
        weighted avg       0.20      0.20      0.20      1694

🏃 View run Naive_Bayes_2 at: http://127.0.0.1:5000/#/experiments/2/runs/fb89216c61be47d98cc9076f735695f6
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/2


In [23]:
import torch
print(torch.__version__)

2.10.0


In [24]:
import torch

if torch.backends.mps.is_available():
    print("Metal GPU (MPS) is available! 🎉")
else:
    print("MPS not available. Check your PyTorch installation version (requires torch >= 1.12).")

MPS not available. Check your PyTorch installation version (requires torch >= 1.12).


In [25]:
import torch
print(torch.backends.mps.is_built())
print(torch.backends.mps.is_available())

False
False
